# 42 LightGBM ウォークフォワード評価 (JP 株式)

| 項目 | 内容 |
|------|------|
| 入力 | `data/equity_jp_ohlcv.parquet`, `data/macro.parquet` |
| 予測ターゲット | `fwd_ret_5d` (5 営業日先のリターン) |
| 評価指標 | Rank IC (スピアマン相関) |
| 出力 | `results/tables/lgbm_models.pkl`, `results/tables/features.csv` |

## 使用特徴量
| カテゴリ | 特徴量 |
|----------|--------|
| リターン | 1/5/20 日リターン、対数リターン |
| 移動平均 | SMA 5/10/20/60, EMA 5/20、価格比 |
| モメンタム | RSI 14/28, MACD |
| ボラティリティ | BB%B, BBW, ATR |
| ボリューム | 出来高比、売買代金 |
| マクロ | USDJPY, 原油, 金, 米 10 年金利 |
| クロスセクション | 日次ランク（相対強度） |

---
## 0. 環境セットアップ

In [ ]:
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import yaml
from scipy.stats import spearmanr
from sklearn.model_selection import TimeSeriesSplit

try:
    import japanize_matplotlib
except ImportError:
    pass

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

print('Setup OK')

---
## 1. 設定・データ読み込み

In [ ]:
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR    = Path(cfg['paths']['data'])
FIGURES_DIR = Path(cfg['paths']['figures'])
TABLES_DIR  = Path(cfg['paths']['tables'])
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = cfg.get('random_seed', 42)
np.random.seed(RANDOM_SEED)

TARGET    = cfg['equity']['target']       # fwd_ret_5d
N_SPLITS  = cfg['model']['n_splits']      # 5
LGB_PARAMS = cfg['model']['params']

print(f'Target  : {TARGET}')
print(f'Splits  : {N_SPLITS}')
print(f'Tables  : {TABLES_DIR.resolve()}')

In [ ]:
df_raw   = pd.read_parquet(DATA_DIR / 'equity_jp_ohlcv.parquet')
df_macro = pd.read_parquet(DATA_DIR / 'macro.parquet')

print(f'OHLCV  : {len(df_raw):,} 行  {df_raw["symbol"].nunique()} 銘柄')
print(f'Macro  : {len(df_macro):,} 行  {df_macro.shape[1]-1} 指標')
print(f'OHLCV 期間: {df_raw["date"].min().date()} ~ {df_raw["date"].max().date()}')
df_raw.head(3)

---
## 2. 特徴量エンジニアリング

In [ ]:
# ---- テクニカル指標計算 ----

def add_returns(df, windows=(1, 5, 20)):
    for w in windows:
        df[f'ret_{w}d']    = df.groupby('symbol')['close'].pct_change(w)
        df[f'logret_{w}d'] = np.log(df.groupby('symbol')['close'].transform(lambda x: x / x.shift(w)))
    return df

def add_forward_returns(df, windows=(1, 5, 20)):
    for w in windows:
        df[f'fwd_ret_{w}d'] = df.groupby('symbol')['close'].transform(
            lambda x: x.pct_change(w).shift(-w))
    return df

def add_sma(df, windows=(5, 10, 20, 60)):
    for w in windows:
        df[f'sma_{w}'] = df.groupby('symbol')['close'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'ema_{w}'] = df.groupby('symbol')['close'].transform(lambda x: x.ewm(span=w, adjust=False).mean())
    for w in windows:
        df[f'price_sma{w}_ratio'] = df['close'] / df[f'sma_{w}']
    return df

def add_rsi(df, windows=(14, 28)):
    for w in windows:
        delta = df.groupby('symbol')['close'].transform(lambda x: x.diff())
        gain  = delta.where(delta > 0, 0.0)
        loss  = -delta.where(delta < 0, 0.0)
        ag    = gain.groupby(df['symbol']).transform(lambda x: x.ewm(com=w-1, min_periods=w).mean())
        al    = loss.groupby(df['symbol']).transform(lambda x: x.ewm(com=w-1, min_periods=w).mean())
        rs    = ag / al.replace(0, np.nan)
        df[f'rsi_{w}'] = 100 - (100 / (1 + rs))
    return df

def add_macd(df, fast=12, slow=26, signal=9):
    def _calc(x):
        ef = x.ewm(span=fast, adjust=False).mean()
        es = x.ewm(span=slow, adjust=False).mean()
        m  = ef - es
        s  = m.ewm(span=signal, adjust=False).mean()
        return pd.DataFrame({'macd': m, 'macd_signal': s, 'macd_hist': m - s})
    result = df.groupby('symbol')['close'].apply(_calc).reset_index(level=0, drop=True)
    return pd.concat([df, result], axis=1)

def add_bollinger(df, window=20, k=2.0):
    df['bb_mid']   = df.groupby('symbol')['close'].transform(lambda x: x.rolling(window, min_periods=window//2).mean())
    df['bb_std']   = df.groupby('symbol')['close'].transform(lambda x: x.rolling(window, min_periods=window//2).std())
    df['bb_upper'] = df['bb_mid'] + k * df['bb_std']
    df['bb_lower'] = df['bb_mid'] - k * df['bb_std']
    band = (df['bb_upper'] - df['bb_lower']).replace(0, np.nan)
    df['bb_pct']   = (df['close'] - df['bb_lower']) / band
    df['bb_width'] = band / df['bb_mid']
    return df

def add_volume(df):
    df['vol_change']  = df.groupby('symbol')['volume'].transform(lambda x: x.pct_change())
    df['vol_sma20']   = df.groupby('symbol')['volume'].transform(lambda x: x.rolling(20, min_periods=5).mean())
    df['vol_ratio']   = df['volume'] / df['vol_sma20'].replace(0, np.nan)
    df['turnover']    = df['close'] * df['volume']
    df['turnover_sma5'] = df.groupby('symbol')['turnover'].transform(lambda x: x.rolling(5, min_periods=2).mean())
    return df

def add_volatility(df, windows=(5, 20)):
    lr = np.log(df.groupby('symbol')['close'].transform(lambda x: x / x.shift(1)))
    for w in windows:
        df[f'vol_{w}d'] = lr.groupby(df['symbol']).transform(
            lambda x: x.rolling(w, min_periods=w//2).std() * np.sqrt(252))
    return df

def add_cs_rank(df, cols):
    for col in cols:
        if col in df.columns:
            df[f'{col}_rank'] = df.groupby('date')[col].rank(pct=True)
    return df

def merge_macro(df, df_m):
    df = df.merge(df_m, on='date', how='left')
    macro_cols = [c for c in df_m.columns if c != 'date']
    df[macro_cols] = df[macro_cols].ffill()
    for col in macro_cols:
        df[f'{col}_ret1d'] = df[col].pct_change()
        df[f'{col}_ret5d'] = df[col].pct_change(5)
    return df

print('関数定義 OK')

In [ ]:
print('特徴量構築中...')
df = df_raw.copy().sort_values(['symbol', 'date']).reset_index(drop=True)
df = add_returns(df)
df = add_forward_returns(df)
df = add_sma(df)
df = add_rsi(df)
df = add_macd(df)
df = add_bollinger(df)
df = add_volume(df)
df = add_volatility(df)
df = add_cs_rank(df, ['ret_1d', 'ret_5d', 'ret_20d', 'vol_20d', 'rsi_14', 'vol_ratio'])
df = merge_macro(df, df_macro)

print(f'特徴量構築完了: {df.shape[0]:,} 行 x {df.shape[1]} 列')
df.head(2)

---
## 3. 特徴量定義・学習/テスト分割

In [ ]:
# 特徴量リスト（フォワードリターンは除外）
EXCLUDE = {'symbol', 'asset_class', 'date', 'open', 'high', 'low', 'close', 'volume'}
EXCLUDE.update({c for c in df.columns if c.startswith('fwd_ret_')})
FEATURES = [c for c in df.columns if c not in EXCLUDE]

# 欠損を除いた学習データ
df_model = df.dropna(subset=FEATURES + [TARGET]).copy()

SPLIT_DATE = cfg['period']['train_end'][:4] + '-' + cfg['period']['train_end'][4:] + '-01'
train = df_model[df_model['date'] < SPLIT_DATE]
test  = df_model[df_model['date'] >= SPLIT_DATE]

print(f'特徴量数  : {len(FEATURES)}')
print(f'学習期間  : {train["date"].min().date()} ~ {train["date"].max().date()}  ({len(train):,} 行)')
print(f'テスト期間: {test["date"].min().date()} ~ {test["date"].max().date()}  ({len(test):,} 行)')
print(f'ターゲット: {TARGET}')

In [ ]:
# ターゲット分布確認
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train[TARGET].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white', density=True)
axes[0].set_title(f'{TARGET} 分布 (学習データ)')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)

train[TARGET].expanding().mean().plot(ax=axes[1], color='steelblue')
axes[1].set_title(f'{TARGET} 累積平均 (学習データ)')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)

plt.suptitle('ターゲット変数の特性確認', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Mean: {train[TARGET].mean():.4f}  Std: {train[TARGET].std():.4f}')
print(f'Up rate: {(train[TARGET] > 0).mean():.2%}')

---
## 4. ウォークフォワード学習

In [ ]:
def rank_ic(y_true, y_pred):
    corr, _ = spearmanr(y_true, y_pred)
    return float(corr)


def walkforward_train(df_all, features, target, n_splits=5, params=None):
    if params is None:
        params = LGB_PARAMS.copy()

    df_all = df_all.dropna(subset=features + [target]).copy()
    df_all = df_all.sort_values('date').reset_index(drop=True)
    dates  = df_all['date'].sort_values().unique()
    tscv   = TimeSeriesSplit(n_splits=n_splits)

    models, metrics_rows, oof_records = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(dates)):
        tr_dates = dates[tr_idx]
        va_dates = dates[va_idx]
        tr_mask  = df_all['date'].isin(tr_dates)
        va_mask  = df_all['date'].isin(va_dates)

        X_tr, y_tr = df_all.loc[tr_mask, features], df_all.loc[tr_mask, target]
        X_va, y_va = df_all.loc[va_mask, features], df_all.loc[va_mask, target]

        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)])

        y_pred = m.predict(X_va)
        ic = rank_ic(y_va.values, y_pred)
        models.append(m)

        metrics_rows.append({
            'fold': fold + 1,
            'val_start': pd.Timestamp(va_dates.min()),
            'val_end':   pd.Timestamp(va_dates.max()),
            'rank_ic':   ic,
            'n_train':   int(tr_mask.sum()),
            'n_val':     int(va_mask.sum()),
        })
        oof = df_all.loc[va_mask, ['date', 'symbol', target]].copy()
        oof['pred'] = y_pred
        oof_records.append(oof)

        print(f'  Fold {fold+1}: RankIC={ic:+.4f}  '
              f'Val {pd.Timestamp(va_dates.min()).date()} ~ {pd.Timestamp(va_dates.max()).date()}')

    metrics_df = pd.DataFrame(metrics_rows)
    oof_df     = pd.concat(oof_records, ignore_index=True)
    print(f'\n  平均 RankIC: {metrics_df["rank_ic"].mean():+.4f} ± {metrics_df["rank_ic"].std():.4f}')
    return models, metrics_df, oof_df


print('=== ウォークフォワード学習開始 ===')
models, metrics_df, oof_df = walkforward_train(df_model, FEATURES, TARGET, n_splits=N_SPLITS)

---
## 5. 評価: Rank IC 推移

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(metrics_df['fold'], metrics_df['rank_ic'],
       color=['#2ca02c' if v > 0 else '#d62728' for v in metrics_df['rank_ic']],
       edgecolor='white', alpha=0.85)
mean_ic = metrics_df['rank_ic'].mean()
ax.axhline(0,       color='black', linewidth=1)
ax.axhline(0.03,    color='green',  linewidth=1.5, linestyle='--', label='目標 IC=0.03')
ax.axhline(mean_ic, color='orange', linewidth=2,   linestyle='-',  label=f'平均 IC={mean_ic:+.4f}')
ax.set_xlabel('Fold')
ax.set_ylabel('Rank IC (Spearman)')
ax.set_title('ウォークフォワード Rank IC')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
path = FIGURES_DIR / '42_rank_ic_by_fold.png'
plt.savefig(path, dpi=120, bbox_inches='tight')
plt.show()

print(metrics_df[['fold', 'val_start', 'val_end', 'rank_ic', 'n_train', 'n_val']].to_string(index=False))

In [ ]:
# OOF の月次 Rank IC
oof_df['ym'] = oof_df['date'].dt.to_period('M')
monthly_ic = oof_df.groupby('ym').apply(
    lambda g: rank_ic(g[TARGET].values, g['pred'].values)
).rename('rank_ic').reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#2ca02c' if v > 0 else '#d62728' for v in monthly_ic['rank_ic']]
ax.bar(range(len(monthly_ic)), monthly_ic['rank_ic'], color=colors, alpha=0.8)
ax.axhline(0, color='black', linewidth=1)
ax.axhline(monthly_ic['rank_ic'].mean(), color='orange', linewidth=1.5, linestyle='--',
           label=f'平均={monthly_ic["rank_ic"].mean():+.4f}')
ax.set_xticks(range(0, len(monthly_ic), max(1, len(monthly_ic)//12)))
ax.set_xticklabels(
    [str(monthly_ic['ym'].iloc[i]) for i in range(0, len(monthly_ic), max(1, len(monthly_ic)//12))],
    rotation=45, ha='right', fontsize=8)
ax.set_title('月次 Rank IC (OOF)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '42_monthly_rank_ic.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. 特徴量重要度

In [ ]:
# 全 Fold の平均重要度
imp_list = [pd.Series(m.feature_importances_, index=FEATURES, name=f'fold{i+1}')
            for i, m in enumerate(models)]
imp_df = pd.concat(imp_list, axis=1)
imp_df['mean_importance'] = imp_df.mean(axis=1)
imp_df = imp_df.sort_values('mean_importance', ascending=False).reset_index()
imp_df.columns = ['feature'] + [f'fold{i+1}' for i in range(len(models))] + ['mean_importance']

TOP_N = 25
top_imp = imp_df.head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_imp['feature'][::-1], top_imp['mean_importance'][::-1], color='steelblue', alpha=0.85)
ax.set_title(f'特徴量重要度 Top {TOP_N} (平均)')
ax.set_xlabel('Importance')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '42_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print(top_imp[['feature', 'mean_importance']].head(20).to_string(index=False))

---
## 7. モデル・特徴量保存

In [ ]:
# モデル保存
model_path = TABLES_DIR / 'lgbm_models.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({'models': models, 'feature_cols': FEATURES}, f)
print(f'モデル保存: {model_path}')

# 特徴量リスト保存
feat_path = TABLES_DIR / 'features.csv'
pd.DataFrame({'feature': FEATURES}).to_csv(feat_path, index=False, encoding='utf-8-sig')
print(f'特徴量リスト保存: {feat_path}')

# OOF 予測値保存
oof_path = TABLES_DIR / 'oof_predictions.csv'
oof_df.to_csv(oof_path, index=False, encoding='utf-8-sig')
print(f'OOF 予測値保存: {oof_path}')

# 評価指標保存
metrics_df.to_csv(TABLES_DIR / 'walkforward_metrics.csv', index=False, encoding='utf-8-sig')

print()
print(f'学習完了:  {len(models)} Fold  平均 RankIC={metrics_df["rank_ic"].mean():+.4f}')

---
## 8. 最新日の予測 (Inference)

In [ ]:
# 最新日のデータで予測
latest_date = df['date'].max()
df_latest   = df[df['date'] == latest_date].copy()

avail_feats = [f for f in FEATURES if f in df_latest.columns]
df_latest   = df_latest.dropna(subset=avail_feats)

if df_latest.empty:
    print('最新日のデータが不足しています')
else:
    X_latest = pd.DataFrame(index=df_latest.index, columns=FEATURES, dtype=float)
    for col in avail_feats:
        X_latest[col] = df_latest[col].values

    preds_all = np.array([m.predict(X_latest) for m in models])
    preds_mean = preds_all.mean(axis=0)

    df_pred = df_latest[['symbol']].copy()
    df_pred['predicted_return'] = preds_mean
    df_pred = df_pred.sort_values('predicted_return', ascending=False).reset_index(drop=True)
    df_pred['rank'] = range(1, len(df_pred) + 1)
    df_pred['pred_pct'] = (df_pred['predicted_return'] * 100).round(2)

    print(f'予測基準日: {latest_date.date()}')
    print(f'予測銘柄数: {len(df_pred)}')
    print()
    print('=== 上位 10 銘柄 ===')
    print(df_pred.head(10)[['rank', 'symbol', 'pred_pct']].to_string(index=False))
    print()
    print('=== 下位 5 銘柄 ===')
    print(df_pred.tail(5)[['rank', 'symbol', 'pred_pct']].to_string(index=False))

    # 保存
    pred_path = TABLES_DIR / 'latest_predictions.csv'
    df_pred.to_csv(pred_path, index=False, encoding='utf-8-sig')
    print(f'\n予測保存: {pred_path}')

In [ ]:
if not df_latest.empty and len(df_pred) > 0:
    import matplotlib.patches as mpatches

    top15 = df_pred.head(15)
    colors = ['#2ca02c' if v > 0 else '#d62728' for v in top15['predicted_return']]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top15['symbol'][::-1], top15['pred_pct'][::-1], color=colors[::-1], alpha=0.85)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('予測リターン (%)')
    ax.set_title(f'予測上位 15 銘柄  (as of {latest_date.date()})')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '42_latest_predictions.png', dpi=120, bbox_inches='tight')
    plt.show()